In [1]:
import os
import base64
import json
import random
from openai import OpenAI
import anthropic
import numpy as np
import re
from tqdm import tqdm
import pandas as pd
import time
from word2number import w2n
from dotenv import load_dotenv


# Ablation Study Configuration

In [2]:
# Ablation Study Configuration
# Choose which agents to enable/disable for the ablation study
ENABLE_VISUAL_AGENT = False    
ENABLE_LANGUAGE_AGENT = True  
ENABLE_HALLUCINATION_AGENT = True 

# Output file suffix for the ablation configuration
ablation_config = []
if ENABLE_VISUAL_AGENT:
    ablation_config.append("visual")
if ENABLE_LANGUAGE_AGENT:
    ablation_config.append("language")
if ENABLE_HALLUCINATION_AGENT:
    ablation_config.append("hallucination")

# Create name suffix based on enabled agents
config_suffix = "_".join(ablation_config)
print(f"Running with configuration: {config_suffix}")

Running with configuration: language_hallucination


# Load dataset

In [3]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "simpsons")

# Set paths relative to base_dir
annotation_path = os.path.join(base_dir, "v1_Annotation_Val_simpsons_vqa.json")
question_path = os.path.join(base_dir, "v1_Question_Val_simpsons_vqa.json")
images_dir = os.path.join(base_dir, "val_images")

def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select high-quality QA pairs (overall_scores == 1.0)
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: annotation['answer']
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [question for question in questions if question['id'] in question_id_to_answer]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.005, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_truth_answers = [
            question_id_to_answer[q['id']]
            for q in sampled_questions
        ]
        # Add image_base64 to each sampled question
        for q in sampled_questions:
            image_relative_path = q['img_path']
            image_path = os.path.join(images_dir, image_relative_path)
            q['image_base64'] = encode_image(image_path)

        return sampled_questions, sampled_truth_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            image_base64 = base64.b64encode(image_file.read()).decode('utf-8')
            return image_base64

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

# Initialize results list
results_ablation = []

# Agents Configuration

In [4]:
load_dotenv()
# Configuration
# MODEL_NAME = "claude-3-5-haiku-20241022"
MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Agent Implementations

# Visual agent: handles image-related tasks, and outputs image description
def visual_agent(image_base64, max_retries=3, retry_delay=2):
    # If visual agent is disabled, return a basic placeholder
    if not ENABLE_VISUAL_AGENT:
        return "This is a cartoon image from The Simpsons."

    prompt = """
    As a cartoon visual expert, describe the image concisely and accurately.

    Guidelines: Analyze key cartoon elements including character design, facial expressions, compositional framing, color palette, and scene semantics.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url",
                                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                visual_desc = completion.choices[0].message.content.strip()
                return visual_desc
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                visual_desc = completion.content[0].text.strip()
                return visual_desc

        except Exception as e:
            print(f"Visual agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Visual agent failed to process the image")
    return None

# Language agent: handles text-related tasks, and outputs initial predicted answer
def language_agent(question, image_base64, visual_desc, max_retries=3, retry_delay=2):
    # If language agent is disabled, return None
    if not ENABLE_LANGUAGE_AGENT:
        return None
    # Note: visual_desc is always available regardless of whether visual agent is enabled or not
    # - If visual agent is enabled: visual_desc contains the generated description
    # - If visual agent is disabled: visual_desc contains "This is a cartoon image from The Simpsons."
    prompt = f"""
    As a cartoon language expert, answer the question based on the provided context using EXACTLY ONE WORD:

    Input:
    Question: {question}
    Visual Description: {visual_desc}

    Guidelines: No explanations or punctuation allowed.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                                }
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64", "media_type": "image/jpeg", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.content[0].text.strip().lower()

            # Process the response
            # 1. Remove punctuation marks
            initial_predicted_answer = initial_predicted_answer.rstrip('.!?')

            # 2. Split into words and get the first word
            words = initial_predicted_answer.split()
            if not words:
                continue

            initial_predicted_answer = words[0]

            # 3. Convert numbers if applicable
            try:
                # Check if word represents a number
                number = w2n.word_to_num(initial_predicted_answer)
                initial_predicted_answer = str(number)
            except ValueError:
                # Check if contains numeric digits
                matches = re.findall(r'\d+', initial_predicted_answer)
                if matches:
                    initial_predicted_answer = matches[0]

            return initial_predicted_answer

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Language agent failed to generate an answer")
    return None

# Hallucination detection agent: implements a critic without verbose output
def hallucination_agent(question, image_base64, initial_predicted_answer, visual_desc, max_retries=3, retry_delay=2):
    # If hallucination agent is disabled, return the initial prediction
    if not ENABLE_HALLUCINATION_AGENT:
        return initial_predicted_answer
    
    # Handle case when language_agent failed or was disabled
    if initial_predicted_answer is None:
        return "unknown"
    
    # First round: Critic evaluates the initial answer
    prompt_first_round = f"""
    As an expert cartoon visual critic, evaluate the accuracy of the initial answer to this visual question.
    
    Question: {question}
    Initial Answer: {initial_predicted_answer}
    Visual Description: {visual_desc}
    
    EVALUATION FRAMEWORK:
    
    1. UNDERSTAND THE QUESTION REQUIREMENTS
    2. EXAMINE VISUAL EVIDENCE
    3. EVALUATE THE ANSWER'S ACCURACY using this scale:
       - 1.0: Completely correct answer that addresses the question perfectly
       - 0.75: Mostly correct with only minor differences
       - 0.5: Partially correct - contains some correct elements but misses important aspects
       - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
       - 0.0: Completely incorrect
    
    YOUR RESPONSE MUST USE THIS FORMAT:
    ACCURACY: [1.0/0.75/0.5/0.25/0.0]
    EXPLANATION: [brief explanation of why the answer is appropriate or not]
    RECOMMENDATION: [KEEP/REVISE]
    CONFIDENCE: [HIGH/LOW] (HIGH if accuracy is 0.75 or higher, LOW otherwise)
    CORRECTION: [suggest a better answer if recommendation is REVISE, otherwise write 'none']
    """
    
    try:
        if is_openai_model:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_first_round},
                        {"type": "image_url", 
                         "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=300,
                temperature=0.1,
            )
            first_response = completion.choices[0].message.content.strip()
        else:
            completion = client.messages.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_first_round},
                        {"type": "image",
                         "source": {
                             "type": "base64",
                             "media_type": "image/jpeg",
                             "data": image_base64
                         }}
                    ]
                }],
                max_tokens=300,
                temperature=0.1,
            )
            first_response = completion.content[0].text.strip()
        
        # Parse the critic's evaluation silently without printing details
        accuracy_match = re.search(r'ACCURACY:\s*(1\.0|0\.75|0\.5|0\.25|0\.0)', first_response, re.IGNORECASE)
        explanation_match = re.search(r'EXPLANATION:\s*(.+?)(?=\n(RECOMMENDATION|CONFIDENCE|CORRECTION)|$)', first_response, re.IGNORECASE | re.DOTALL)
        recommendation_match = re.search(r'RECOMMENDATION:\s*(KEEP|REVISE)', first_response, re.IGNORECASE)
        confidence_match = re.search(r'CONFIDENCE:\s*(HIGH|LOW)', first_response, re.IGNORECASE)
        correction_match = re.search(r'CORRECTION:\s*(.+?)(?=\n|$)', first_response, re.IGNORECASE | re.DOTALL)
        
        # Extract values with fallbacks
        accuracy = 0.5 if not accuracy_match else float(accuracy_match.group(1))
        explanation = "No explanation provided" if not explanation_match else explanation_match.group(1).strip()
        recommendation = "KEEP" if not recommendation_match else recommendation_match.group(1).upper()
        confidence = "LOW" if accuracy < 0.75 else "HIGH"
        if confidence_match:
            confidence = confidence_match.group(1).upper()
        correction = initial_predicted_answer if not correction_match or correction_match.group(1).strip().lower() == "none" else correction_match.group(1).strip()
        
        # If recommendation is KEEP, just return the initial answer with confidence marker if needed
        if recommendation == "KEEP":
            if confidence == "LOW":
                return f"[LOW CONFIDENCE] {initial_predicted_answer.lower().strip().rstrip('.!?')}"
            return initial_predicted_answer.lower().strip().rstrip('.!?')
        
        # If recommendation is REVISE, go for second round evaluation
        prompt_second_round = f"""
        As an expert cartoon visual validator, make a final decision about the answer to this question.
        
        Question: {question}
        Initial Answer: {initial_predicted_answer}
        Critic's Assessment: {explanation}
        Suggested Correction: {correction}
        
        Based on the image, provide the MOST ACCURATE single-word answer.
        
        YOUR RESPONSE FORMAT:
        FINAL_ANSWER: [single word answer]
        CONFIDENCE: [HIGH/LOW]
        """
        
        if is_openai_model:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_second_round},
                        {"type": "image_url", 
                         "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                    ]
                }],
                max_tokens=150,
                temperature=0.1,
            )
            second_response = completion.choices[0].message.content.strip()
        else:
            completion = client.messages.create(
                model=MODEL_NAME,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_second_round},
                        {"type": "image",
                         "source": {
                             "type": "base64",
                             "media_type": "image/jpeg",
                             "data": image_base64
                         }}
                    ]
                }],
                max_tokens=150,
                temperature=0.1,
            )
            second_response = completion.content[0].text.strip()
        
        # Parse the second response silently
        final_answer_match = re.search(r'FINAL_ANSWER:\s*(.+?)(?=\n|$)', second_response, re.IGNORECASE)
        final_confidence_match = re.search(r'CONFIDENCE:\s*(HIGH|LOW)', second_response, re.IGNORECASE)
        
        final_answer = correction if not final_answer_match else final_answer_match.group(1).strip()
        final_confidence = confidence if not final_confidence_match else final_confidence_match.group(1).upper()
        
        # Format and return the final answer
        final_answer = final_answer.lower().strip().rstrip('.!?')
        if final_confidence == "LOW":
            return f"[LOW CONFIDENCE] {final_answer}"
        return final_answer
        
    except Exception as e:
        # Silent error handling, just write to log without detailed output
        print(f"Hallucination agent error: {str(e)[:100]}..." if len(str(e)) > 100 else f"Hallucination agent error: {e}")
        # Fallback to initial answer with low confidence marker
        return f"[LOW CONFIDENCE] {initial_predicted_answer.lower().strip().rstrip('.!?')}"

Using OpenAI model: gpt-4o-mini


# Calculate accuracy

In [5]:
def compute_accuracy(question, truth_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2):
    if predicted_answer is None:
        return 0

    prompt = f"""
    Evaluate the accuracy of the predicted answer according to criteria below.

    Input:
    Question: {question}
    Answer type: {answer_type}
    True answer: {truth_answer}
    Predicted answer: {predicted_answer}
    
    Evaluation Rules:
    1. Focus on whether the predicted answer correctly captures the core information required by the question.
    2. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
    3. Do NOT include any explanation, words, punctuation, or additional content.
    4. Answer Type Considerations:
    - Yes/No questions: Check if the meaning is equivalent
    - Number questions: Verify numerical accuracy
    - Other questions: Check for key information match

    Scoring Criteria:
    - 1.0: Contains the correct answer with the same core meaning, even if phrased differently or with additional details
    - 0.75: Mostly correct with only minor differences that slightly change the meaning
    - 0.5: Partially correct - contains some correct elements but misses important aspects
    - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
    - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering by claiming insufficient information
    """
    
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.content[0].text.strip()

            numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
            if numeric_match:
                score = float(numeric_match.group(1))
            else:
                score = 0.0

            return score

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    # Return 0 if it can't parse the score
    return 0.0

# Evaluate model performance

In [6]:
try:
    # Initialize counters
    correct_count = 0
    total_count = 0
    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data
    sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Initialize results storage
    accuracies = []

    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for i, (question, truth_answer) in enumerate(tqdm(zip(sampled_questions, sampled_truth_answers),
                                     total=len(sampled_questions))):
        try:
            question_id = question['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question_text = question['question']
            image_relative_path = question['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            print(f"\nProcessing question {i + 1}/{len(sampled_questions)}: ID {question_id}")

            # Build image path and encode
            image_path = os.path.join(images_dir, image_relative_path)
            image_base64 = encode_image(image_path)

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to image encoding failure")
                continue
            
            # Multi agent processing based on configuration
            # Step 1: Visual agent (may be disabled)
            visual_desc = visual_agent(image_base64)
            if visual_desc is None:
                print(f"Skipping question ID {question_id} - Failed to get image description")
                continue

            # Step 2: Language agent (may process directly or with visual description)
            initial_predicted_answer = language_agent(question_text, image_base64, visual_desc)
            
            # If language agent is disabled or failed, use a default answer
            if initial_predicted_answer is None:
                if not ENABLE_LANGUAGE_AGENT:
                    # Default answer when language agent is disabled
                    initial_predicted_answer = "unknown"  
                    print(f"Language agent disabled - using default answer: '{initial_predicted_answer}'")
                else:
                    print(f"Skipping question ID {question_id} - Failed to generate answer")
                    continue

            # Step 3: Hallucination agent (may be disabled)
            # Updated to match the modified hallucination_agent function signature
            final_answer = hallucination_agent(
                question=question_text,
                image_base64=image_base64,
                initial_predicted_answer=initial_predicted_answer,
                visual_desc=visual_desc
            )

            # Use final answer if available, otherwise fallback to initial
            model_answer = final_answer if final_answer else initial_predicted_answer

            # Calculate accuracy
            accuracy = compute_accuracy(
                question=question_text,
                truth_answer=truth_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question_text}")
            print(f"Answer Type: {answer_type}")
            print(f"Truth Answer: {truth_answer}")
            print(f"Predicted Answer: {model_answer}")
            if accuracy is not None:
                accuracies.append(accuracy)
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question_text}")

            # Store result
            result = {
                'question_id': question_id,
                'question': question_text,
                'answer_type': answer_type,
                'truth_answer': truth_answer,
                'predicted_answer': model_answer,
                'accuracy': accuracy
            }
            results_ablation.append(result)

        except Exception as e:
            print(f"Error processing question {question.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy
    if accuracies:
        average_accuracy = np.mean(accuracies)
        print(f"Average Accuracy: {average_accuracy:.4f}")
    else:
        print("No valid accuracy data")
        average_accuracy = 0

    # Save results with configuration in filename using the new directory structure
    safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')
    results_dir = os.path.join(os.getcwd(), "results")
    os.makedirs(results_dir, exist_ok=True)
    
    # Create ablation subdirectory if it doesn't exist
    os.makedirs(os.path.join(results_dir, "ablation"), exist_ok=True)
    
    # Save to ablation subdirectory with configuration in filename (using consistent pattern)
    output_path = os.path.join(results_dir, "ablation", f'simpsons_ablation_{config_suffix}_{safe_model_name}.csv')

    # Check if the file exists and explicitly remove it
    if os.path.exists(output_path):
        try:
            os.remove(output_path)
            print(f"Existing file removed: {output_path}")
        except Exception as e:
            print(f"Error removing existing file: {e}")

    # Add average accuracy as the last row
    average_result = {
        'question_id': 'Average',
        'question': f'Total Questions: {len(processed_question_ids)}',  
        'answer_type': 'All',  
        'truth_answer': '',
        'predicted_answer': '',
        'accuracy': average_accuracy 
    }
    results_ablation.append(average_result)

    column_order = [
        'question_id',
        'question',
        'answer_type',
        'truth_answer',
        'predicted_answer',
        'accuracy'
    ]

    # Convert to DataFrame and save with error handling
    try:
        results_df = pd.DataFrame(results_ablation)
        results_df = results_df[column_order]
        
        # Save with explicit file opening to ensure it closes properly
        results_df.to_csv(output_path, index=False)
        
        # Verify the file was created
        if os.path.exists(output_path):
            print(f"Results successfully saved to: {output_path}")
        else:
            print(f"Warning: File was not created at {output_path}")
    except Exception as e:
        print(f"Error saving results to CSV: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

  0%|          | 0/36 [00:00<?, ?it/s]


Processing question 1/36: ID 77311


  3%|▎         | 1/36 [00:10<06:22, 10.94s/it]

Question ID: 77311
Question: what is on the shelf?
Answer Type: other
Truth Answer: book
Predicted Answer: shelf
Accuracy: 0.2500

Processing question 2/36: ID 12809


  6%|▌         | 2/36 [00:18<04:59,  8.81s/it]

...

Question ID: 71583
Question: what is in the background?
Answer Type: other
Truth Answer: table
Predicted Answer: monopoly
Accuracy: 0.0000

Processing question 35/36: ID 37083


 97%|█████████▋| 35/36 [04:46<00:07,  7.49s/it]

Question ID: 37083
Question: what are the men doing?
Answer Type: other
Truth Answer: standing
Predicted Answer: conversing
Accuracy: 0.5000

Processing question 36/36: ID 96818


100%|██████████| 36/36 [04:54<00:00,  8.17s/it]

Question ID: 96818
Question: what is the person sitting on?
Answer Type: other
Truth Answer: chair
Predicted Answer: chair
Accuracy: 1.0000
Average Accuracy: 0.7639
Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_hallucination_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_hallucination_gpt_4o_mini.csv


# Save results

In [7]:
# Clean up ablation results to remove any existing average rows
results_ablation = [r for r in results_ablation if r['question_id'] != 'Average']

# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in results_ablation))

# Add row numbers to each result
for i, result in enumerate(results_ablation, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_ablation) + 1,
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'truth_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy 
}
results_ablation.append(average_result)

# Define column order with row_num first
column_order = [
    'row_num',
    'question_id',
    'question',
    'answer_type',
    'truth_answer',
    'predicted_answer',
    'accuracy'
]

# Save results with configuration in filename using the new directory structure
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)

# Create ablation subdirectory if it doesn't exist (using consistent pattern)
os.makedirs(os.path.join(results_dir, "ablation"), exist_ok=True)

# Save to ablation subdirectory with configuration in filename (using consistent pattern)
output_path = os.path.join(results_dir, "ablation", f'simpsons_ablation_{config_suffix}_{safe_model_name}.csv')

# Check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(results_ablation)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_hallucination_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/ablation/simpsons_ablation_language_hallucination_gpt_4o_mini.csv
